In [1]:
from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {DATA_DIR}")
print(f"Interim data: {INTERIM_DIR}")
print(f"Reports: {REPORTS_DIR}")

Project root: c:\Projects\creditlens-ai
Raw data: c:\Projects\creditlens-ai\data\raw
Interim data: c:\Projects\creditlens-ai\data\interim
Reports: c:\Projects\creditlens-ai\reports


In [2]:
feature_files = {
    "bureau": (
        INTERIM_DIR /
        "bureau_customer_features.csv"
    ),
    "previous": (
        INTERIM_DIR /
        "previous_application_customer_features.csv"
    ),
    "installments": (
        INTERIM_DIR /
        "installments_customer_features.csv"
    ),
}

for name, path in feature_files.items():
    print(
        f"{name:15} "
        f"exists={path.exists()}"
    )

bureau          exists=True
previous        exists=True
installments    exists=True


In [3]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

bureau_features = pd.read_csv(
    feature_files["bureau"]
)

previous_features = pd.read_csv(
    feature_files["previous"]
)

installment_features = pd.read_csv(
    feature_files["installments"]
)

print(f"Application:  {application_train.shape}")
print(f"Bureau:       {bureau_features.shape}")
print(f"Previous:     {previous_features.shape}")
print(f"Installments: {installment_features.shape}")

Application:  (307511, 122)
Bureau:       (305811, 20)
Previous:     (338857, 24)
Installments: (339587, 30)


In [4]:
train_full = (
    application_train
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

print(f"Merged shape: {train_full.shape}")
print(
    "Duplicate customers:",
    train_full["SK_ID_CURR"].duplicated().sum()
)
print(
    "Missing TARGET:",
    train_full["TARGET"].isna().sum()
)

Merged shape: (307511, 193)
Duplicate customers: 0
Missing TARGET: 0


In [5]:
engineered_flags = pd.DataFrame(
    {
        "BUREAU_HAS_HISTORY": (
            train_full["BUREAU_LOAN_COUNT"]
            .notna()
            .astype("int8")
        ),
        "PREV_HAS_HISTORY": (
            train_full["PREV_APPLICATION_COUNT"]
            .notna()
            .astype("int8")
        ),
        "INST_HAS_HISTORY": (
            train_full["INST_PAYMENT_RECORD_COUNT"]
            .notna()
            .astype("int8")
        ),
        "DAYS_EMPLOYED_ANOMALY": (
            train_full["DAYS_EMPLOYED"] == 365243
        ).astype("int8")
    },
    index=train_full.index
)

train_full = pd.concat(
    [train_full, engineered_flags],
    axis=1
)

train_full.loc[
    train_full["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(
    "Shape after engineered flags:",
    train_full.shape
)

Shape after engineered flags: (307511, 197)


In [6]:
y = train_full["TARGET"].copy()

X = train_full.drop(
    columns=[
        "TARGET",
        "SK_ID_CURR",
        "CODE_GENDER"
    ]
).copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(
    f"Positive class rate: "
    f"{y.mean() * 100:.2f}%"
)

print(
    "CODE_GENDER in X:",
    "CODE_GENDER" in X.columns
)

X shape: (307511, 194)
y shape: (307511,)
Positive class rate: 8.07%
CODE_GENDER in X: False


In [7]:
categorical_columns = (
    X
    .select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

numeric_columns = (
    X
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

for column in categorical_columns:
    X[column] = (
        X[column]
        .astype("object")
        .where(
            X[column].notna(),
            "__MISSING__"
        )
        .astype(str)
    )

print(
    f"Numeric features: "
    f"{len(numeric_columns)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_columns)}"
)

print(
    f"Total features: "
    f"{X.shape[1]}"
)

print(
    "Categorical missing values:",
    X[categorical_columns]
    .isna()
    .sum()
    .sum()
)

Numeric features: 179
Categorical features: 15
Total features: 194
Categorical missing values: 0


In [8]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print(f"Number of folds: {cv.get_n_splits()}")

Number of folds: 3


In [9]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    print(
        f"Fold {fold}: "
        f"train={len(train_idx):,}, "
        f"validation={len(val_idx):,}, "
        f"train positive={y_train_fold.mean():.4f}, "
        f"validation positive={y_val_fold.mean():.4f}"
    )

Fold 1: train=205,007, validation=102,504, train positive=0.0807, validation positive=0.0807
Fold 2: train=205,007, validation=102,504, train positive=0.0807, validation positive=0.0807
Fold 3: train=205,008, validation=102,503, train positive=0.0807, validation positive=0.0807


In [10]:
cv_results = []
oof_probabilities = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    print(f"\n{'=' * 60}")
    print(f"FOLD {fold}")
    print(f"{'=' * 60}")

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",

        iterations=1470,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=5,

        auto_class_weights="Balanced",

        random_seed=42,
        thread_count=-1,

        verbose=250,
        allow_writing_files=False
    )

    start_time = time.time()

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=categorical_columns
    )

    training_time = time.time() - start_time

    probabilities = model.predict_proba(
        X_val_fold
    )[:, 1]

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    oof_probabilities[val_idx] = probabilities

    tn, fp, fn, tp = confusion_matrix(
        y_val_fold,
        predictions,
        labels=[0, 1]
    ).ravel()

    fold_result = {
        "fold": fold,
        "train_size": len(train_idx),
        "validation_size": len(val_idx),
        "accuracy": accuracy_score(
            y_val_fold,
            predictions
        ),
        "precision": precision_score(
            y_val_fold,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_val_fold,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_val_fold,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_val_fold,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_val_fold,
            probabilities
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "training_time_seconds": training_time
    }

    cv_results.append(fold_result)

    print(
        f"\nFold {fold} completed:"
        f"\nROC-AUC = {fold_result['roc_auc']:.6f}"
        f"\nPR-AUC  = {fold_result['pr_auc']:.6f}"
        f"\nF1       = {fold_result['f1']:.6f}"
        f"\nRecall   = {fold_result['recall']:.6f}"
        f"\nTime     = {training_time:.2f} sec"
    )

    del model
    del X_train_fold
    del X_val_fold
    gc.collect()

cv_results = pd.DataFrame(cv_results)


FOLD 1
0:	total: 366ms	remaining: 8m 57s
250:	total: 42.1s	remaining: 3m 24s
500:	total: 1m 25s	remaining: 2m 44s
750:	total: 2m 13s	remaining: 2m 7s
1000:	total: 3m 1s	remaining: 1m 25s
1250:	total: 3m 49s	remaining: 40.2s
1469:	total: 4m 32s	remaining: 0us

Fold 1 completed:
ROC-AUC = 0.776712
PR-AUC  = 0.273886
F1       = 0.301004
Recall   = 0.639637
Time     = 273.02 sec

FOLD 2
0:	total: 182ms	remaining: 4m 27s
250:	total: 48.2s	remaining: 3m 53s
500:	total: 1m 38s	remaining: 3m 10s
750:	total: 2m 29s	remaining: 2m 23s
1000:	total: 3m 18s	remaining: 1m 32s
1250:	total: 4m 3s	remaining: 42.7s
1469:	total: 4m 44s	remaining: 0us

Fold 2 completed:
ROC-AUC = 0.779053
PR-AUC  = 0.267857
F1       = 0.300841
Recall   = 0.646526
Time     = 284.73 sec

FOLD 3
0:	total: 162ms	remaining: 3m 58s
250:	total: 46.2s	remaining: 3m 44s
500:	total: 1m 32s	remaining: 2m 59s
750:	total: 2m 21s	remaining: 2m 15s
1000:	total: 3m 50s	remaining: 1m 47s
1250:	total: 6m 11s	remaining: 1m 5s
1469:	total: 7

In [11]:
cv_results[
    [
        "fold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "training_time_seconds"
    ]
]

,fold,accuracy,precision,recall,f1,roc_auc,pr_auc,training_time_seconds
0,1,0.760175,0.196810,0.639637,0.301004,0.776712,0.273886,273.022933
1,2,0.757405,0.196028,0.646526,0.300841,0.779053,0.267857,284.725164
2,3,0.760827,0.195941,0.632387,0.299182,0.777940,0.263877,456.508150


In [12]:
metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc"
]

cv_summary = pd.DataFrame({
    "metric": metric_columns,
    "mean": [
        cv_results[m].mean()
        for m in metric_columns
    ],
    "std": [
        cv_results[m].std()
        for m in metric_columns
    ],
    "min": [
        cv_results[m].min()
        for m in metric_columns
    ],
    "max": [
        cv_results[m].max()
        for m in metric_columns
    ]
})

cv_summary

,metric,mean,std,min,max
0,accuracy,0.759469,0.001817,0.757405,0.760827
1,precision,0.196260,0.000478,0.195941,0.196810
2,recall,0.639517,0.007070,0.632387,0.646526
3,f1,0.300342,0.001008,0.299182,0.301004
4,roc_auc,0.777902,0.001171,0.776712,0.779053
5,pr_auc,0.268540,0.005039,0.263877,0.273886


In [13]:
oof_predictions = (
    oof_probabilities >= 0.50
).astype(int)

oof_metrics = {
    "accuracy": accuracy_score(
        y,
        oof_predictions
    ),
    "precision": precision_score(
        y,
        oof_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y,
        oof_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y,
        oof_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y,
        oof_probabilities
    ),
    "pr_auc": average_precision_score(
        y,
        oof_probabilities
    )
}

pd.Series(oof_metrics)

accuracy     0.759469
precision    0.196259
recall       0.639517
f1           0.300346
roc_auc      0.777891
pr_auc       0.268364
dtype: float64

In [14]:
cv_fold_results_path = (
    REPORTS_DIR / "catboost_stability_cv_folds.csv"
)

cv_results.to_csv(
    cv_fold_results_path,
    index=False
)

cv_summary_path = (
    REPORTS_DIR / "catboost_stability_cv_summary.csv"
)

cv_summary.to_csv(
    cv_summary_path,
    index=False
)

oof_metrics_path = (
    REPORTS_DIR / "catboost_oof_metrics.csv"
)

pd.DataFrame(
    [oof_metrics]
).to_csv(
    oof_metrics_path,
    index=False
)

print(f"Saved: {cv_fold_results_path}")
print(f"Saved: {cv_summary_path}")
print(f"Saved: {oof_metrics_path}")

Saved: c:\Projects\creditlens-ai\reports\catboost_stability_cv_folds.csv
Saved: c:\Projects\creditlens-ai\reports\catboost_stability_cv_summary.csv
Saved: c:\Projects\creditlens-ai\reports\catboost_oof_metrics.csv


In [15]:
development_metrics = {
    "accuracy": 0.756874,
    "precision": 0.196155,
    "recall": 0.649345,
    "f1": 0.301294,
    "roc_auc": 0.782085,
    "pr_auc": 0.279699
}

validation_comparison = pd.DataFrame({
    "Development_Validation": development_metrics,
    "OOF_3Fold_CV": oof_metrics
}).T

validation_comparison["evaluation_type"] = [
    "Development split",
    "Out-of-fold stability estimate"
]

validation_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc,evaluation_type
Development_Validation,0.756874,0.196155,0.649345,0.301294,0.782085,0.279699,Development split
OOF_3Fold_CV,0.759469,0.196259,0.639517,0.300346,0.777891,0.268364,Out-of-fold stability estimate


In [16]:
validation_delta = pd.DataFrame({
    "metric": list(oof_metrics.keys()),
    "development": [
        development_metrics[m]
        for m in oof_metrics
    ],
    "oof": [
        oof_metrics[m]
        for m in oof_metrics
    ]
})

validation_delta["change"] = (
    validation_delta["oof"]
    - validation_delta["development"]
)

validation_delta

,metric,development,oof,change
0,accuracy,0.756874,0.759469,0.002595
1,precision,0.196155,0.196259,0.000104
2,recall,0.649345,0.639517,-0.009828
3,f1,0.301294,0.300346,-0.000948
4,roc_auc,0.782085,0.777891,-0.004194
5,pr_auc,0.279699,0.268364,-0.011335


In [17]:
validation_comparison_path = (
    REPORTS_DIR /
    "catboost_development_vs_oof.csv"
)

validation_comparison.to_csv(
    validation_comparison_path,
    index=True
)

print(f"Saved: {validation_comparison_path}")

Saved: c:\Projects\creditlens-ai\reports\catboost_development_vs_oof.csv


## Final Validation Summary

The primary research candidate was evaluated using 3-fold stratified cross-validation with `CODE_GENDER` excluded from the predictive feature set.

The CatBoost architecture was fixed before this stability evaluation. Therefore, these results are interpreted as a stability / out-of-fold evaluation rather than a completely independent untouched test-set estimate.

### Cross-Validation Results

Across three folds:

- Mean ROC-AUC: **0.7779 ± 0.0012**
- Mean PR-AUC: **0.2685 ± 0.0050**
- Mean F1 at threshold 0.50: **0.3003 ± 0.0010**
- Mean recall at threshold 0.50: **0.6395 ± 0.0071**

The small variation in ROC-AUC and F1 across folds indicates that model performance is relatively stable across different stratified partitions of the training population.

### Out-of-Fold Performance

Aggregating predictions for all 307,511 observations produced:

- ROC-AUC: **0.7779**
- PR-AUC: **0.2684**
- F1: **0.3003**
- Recall: **0.6395**

The OOF scores are slightly lower than the earlier development-validation result (ROC-AUC 0.7821, PR-AUC 0.2797), providing a more conservative estimate of expected model performance.

These results support the gender-free CatBoost model as the primary research candidate while preserving LightGBM as a substantially faster alternative.

No autonomous lending decision rule or group-specific threshold is selected from this evaluation.